In [1]:
import polars as pl

file_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.inter'
inter = pl.read_csv(file_path, separator='\t', has_header=True, quote_char=None, infer_schema_length=10000)
print(inter.head(10))

shape: (10, 2)
┌───────────────┬─────────────────────────────────┐
│ user_id:token ┆ item_id_list:token_seq          │
│ ---           ┆ ---                             │
│ i64           ┆ str                             │
╞═══════════════╪═════════════════════════════════╡
│ 21545         ┆ 560041 888698 335779 315938 81… │
│ 62305         ┆ 777467 16712 777467 416516 861… │
│ 18714         ┆ 908198 737893 963948 272252 21… │
│ 37294         ┆ 9356 55574 435532 370008 70768… │
│ 21796         ┆ 605813 1012727 316706 332985 9… │
│ 15297         ┆ 215073 656033 578228 1001258 3… │
│ 18471         ┆ 220830 863383 329495 171967 38… │
│ 21545         ┆ 797000 195085 134298 887927 10… │
│ 10305         ┆ 151990 270793 882600 1045001 7… │
│ 76725         ┆ 421830 741944 59483 421830 494… │
└───────────────┴─────────────────────────────────┘


In [2]:
import json

code_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.index.json'
with open(code_path, 'r') as f:
    items2codes = json.load(f)
print(items2codes['0'])

['<|a_107|>', '<|b_1|>', '<|c_501|>', '<|d_460|>']


In [3]:
inter_item_id = inter.select("item_id_list:token_seq")


In [7]:
sample = inter_item_id[11]['item_id_list:token_seq']
for i in sample:
    print(i)


1027338 494124 497235 492546 511137 883363 540113 423181 807565 1034133 924439 900861 889701 186931 379919 130388 488406 487010 411628 56537


In [9]:
from tqdm.auto import tqdm
import json

# 1. 预处理 mapping
item_code_map = {k: "".join(v) for k, v in items2codes.items()}

# 2. 定义带有进度条的转换函数
# 使用 tqdm 监控处理进度
total_rows = inter.height
pbar = tqdm(total=total_rows, desc="Processing rows")

def transform_seq(seq_str):
    pbar.update(1)
    if not seq_str:
        return ""
    return ",".join((item_code_map[x] for x in seq_str.split()))

try:
    # 处理数据
    df_processed = inter.select(
        pl.col('item_id_list:token_seq')
        .map_elements(transform_seq, return_dtype=pl.String)
        .alias('text')
    )
    
    # 保存为 .jsonl 格式 (NDJSON)
    # 这种格式每行一个 JSON 对象，方便流式读取和检查，且不会像缩进 JSON 那样占用过多空间
    output_file = 'llama_factory_data.jsonl'
    print(f"Saving to {output_file} in JSONL format...")
    df_processed.write_ndjson(output_file)
    print("Done.")
        
finally:
    pbar.close()

Processing rows: 100%|█████████▉| 22517817/22520616 [22:04<00:00, 31026.22it/s] 

Saving to llama_factory_data.jsonl in JSONL format...


Processing rows: 100%|██████████| 22520616/22520616 [26:38<00:00, 14084.51it/s]

Done.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, pre_tokenizers, processors, Regex, decoders
import os
import json

class TokenExtender:
    def __init__(self, data_path, dataset, index_file=".index.json"):
        self.data_path = data_path
        self.dataset = dataset
        self.index_file = index_file
        self.indices = None
        self.new_tokens = None
        
    def _load_data(self):
        with open(os.path.join(self.data_path, self.dataset + self.index_file), 'r') as f:
            self.indices = json.load(f)
    
    def get_new_tokens(self):
        if self.new_tokens is not None:
            return self.new_tokens
            
        if self.indices is None:
            self._load_data()
        
        self.new_tokens = set()
        for index in self.indices.values():
            for token in index:
                self.new_tokens.add(token)
        self.new_tokens = sorted(list(self.new_tokens))
        
        return self.new_tokens

# 1. 定义输入和输出路径
model_path = '/home/hongminjie/models/Qwen3-1.7B'
output_dir = './yambda/model' 

print(f"Loading model from {model_path}")
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)

# 2. 加载新 Token
sid_index_path = "./yambda/sequential-multievent-500m/sequential-multievent-500m.index.json"
print(f"Loading index from {sid_index_path}")

token_extender = TokenExtender(
    data_path=os.path.dirname(sid_index_path),
    dataset=os.path.basename(sid_index_path).split('.')[0]
)
new_tokens = token_extender.get_new_tokens()

if new_tokens:
    print(f"Found {len(new_tokens)} tokens from dataset")
    
    # 3. 创建新的词表，只包含数据集中的token和必要的特殊token
    special_tokens_list = ['<|pad|>', '<|eos|>', '<|bos|>', '<|unk|>', ',']
    
    # 合并：特殊token在前，数据token在后
    all_tokens = special_tokens_list + new_tokens
    vocab = {token: idx for idx, token in enumerate(all_tokens)}
    
    print(f"Creating new tokenizer with {len(vocab)} tokens (including {len(special_tokens_list)} special tokens)")
    
    # 4. 使用 tokenizers 库创建一个基于词表的 tokenizer
    tokenizer_backend = Tokenizer(models.WordLevel(vocab=vocab, unk_token='<|unk|>'))
    
    # 设置预处理器：使用正则表达式匹配 <|...|> 格式的token和逗号
    tokenizer_backend.pre_tokenizer = pre_tokenizers.Split(
        pattern=Regex(r'(<\|[^|]+\|>|,)'),
        behavior='isolated',
        invert=False
    )
    
    # 设置decoder：移除tokens之间的空格
    tokenizer_backend.decoder = decoders.Replace(" ", "")
    
    # 包装成 HuggingFace tokenizer
    new_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_backend,
        pad_token='<|pad|>',
        eos_token='<|eos|>',
        bos_token='<|bos|>',
        unk_token='<|unk|>',
    )
    
    print(f"New tokenizer vocab size: {len(new_tokenizer)}")
    
    # 5. 调整模型 embedding 大小以匹配新词表
    print("Resizing model embeddings...")
    model.resize_token_embeddings(len(new_tokenizer))
    
    # 6. 验证
    test_text = "<|a_60|><|b_419|><|c_250|><|d_69|>,<|a_260|><|b_304|><|c_23|><|d_445|>,<|a_127|><|b_236|><|c_120|><|d_24|>"
    encoded = new_tokenizer.tokenize(test_text)
    token_ids = new_tokenizer.encode(test_text, add_special_tokens=False)
    decoded = new_tokenizer.decode(token_ids)
    
    print(f"\nVerification:")
    print(f"  Input length: {len(test_text)} chars")
    print(f"  Number of tokens: {len(encoded)}")
    print(f"  First 10 tokens: {encoded[:10]}")
    print(f"  Token IDs (first 10): {token_ids[:10]}")
    print(f"  Decoded length: {len(decoded)} chars")
    print(f"  Match original: {decoded == test_text}")
    if decoded != test_text:
        print(f"  Input:   {test_text[:50]}...")
        print(f"  Decoded: {decoded[:50]}...")
    
    # 7. 保存
    print(f"\nSaving model and tokenizer to {output_dir}...")
    model.save_pretrained(output_dir)
    new_tokenizer.save_pretrained(output_dir)
    print("Done.")
    
    # 打印词表信息
    print(f"\nTokenizer special tokens:")
    print(f"  PAD: {new_tokenizer.pad_token} (ID: {new_tokenizer.pad_token_id})")
    print(f"  EOS: {new_tokenizer.eos_token} (ID: {new_tokenizer.eos_token_id})")
    print(f"  BOS: {new_tokenizer.bos_token} (ID: {new_tokenizer.bos_token_id})")
    print(f"  UNK: {new_tokenizer.unk_token} (ID: {new_tokenizer.unk_token_id})")
else:
    print("No tokens found to add.")

Loading model from /home/hongminjie/models/Qwen3-1.7B


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.77it/s]


Loading index from ./yambda/sequential-multievent-500m/sequential-multievent-500m.index.json
Found 2309 tokens from dataset
Creating new tokenizer with 2314 tokens (including 5 special tokens)
New tokenizer vocab size: 2314
Resizing model embeddings...

Verification:
  Input length: 106 chars
  Number of tokens: 14
  First 10 tokens: ['<|a_60|>', '<|b_419|>', '<|c_250|>', '<|d_69|>', ',', '<|a_260|>', '<|b_304|>', '<|c_23|>', '<|d_445|>', ',']
  Token IDs (first 10): [473, 870, 1195, 2018, 4, 182, 743, 1183, 1923, 4]
  Decoded length: 106 chars
  Match original: True

Saving model and tokenizer to ./yambda/model...
Done.

Tokenizer special tokens:
  PAD: <|pad|> (ID: 0)
  EOS: <|eos|> (ID: 1)
  BOS: <|bos|> (ID: 2)
  UNK: <|unk|> (ID: 3)


In [13]:
print(tokenizer.eos_token_id)
print(tokenizer.pad_token_id)

151645
151643
